# Yaya-125M Training (Kaggle)
**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100) → Save

Outputs (checkpoints) are saved to `/kaggle/working/` automatically.
Download them after the session ends via the Output tab.

In [ ]:
# Step 1: Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
if torch.cuda.device_count() > 1:
    print(f'  ({torch.cuda.device_count()} GPUs available)')

In [ ]:
# Step 2: Setup
import os
!git clone https://github.com/jaylink-coder/miss-yaya.git /kaggle/working/miss-yaya 2>/dev/null || \
    (cd /kaggle/working/miss-yaya && git pull)
!pip install -q sentencepiece datasets pyyaml
os.chdir('/kaggle/working/miss-yaya/yaya-ai')
print('Working dir:', os.getcwd())

In [ ]:
# Step 3: Pretrain
# Checkpoints go to /kaggle/working/yaya-checkpoints
# Download from the Output tab when done (or between sessions)
!python scripts/kaggle_run.py

In [ ]:
# Step 4: SFT (run after pretraining)
!python scripts/kaggle_run_sft.py

In [ ]:
# Step 5: Evaluate
import glob
sft_ckpts = sorted(glob.glob('/kaggle/working/yaya-sft-checkpoints/checkpoint-*'))
best = sft_ckpts[-1] if sft_ckpts else None
if best:
    import os
    os.system(
        f'python scripts/eval_yaya.py '
        f'--model_config configs/model/yaya_125m.yaml '
        f'--checkpoint {best} --chat'
    )
else:
    print('No SFT checkpoint found.')